# Assignment 03: Parsing & Grammar Analysis

**ชื่อ-นามสกุล:** ภูริพัฒน์ เหมกุล  
**รหัสนักศึกษา:** 6752300313 

1. Create an ambiguous English sentence with at least two meanings.
แต่งประโยคภาษาอังกฤษที่มีความกำกวม (แปลได้มากกว่า 1 ความหมาย) มา 1 ประโยค

2. Write a CFG that accounts for all interpretations of the sentence.
เขียนกฎ CFG รองรับประโยคที่แต่งขึ้นมาให้ครอบคลุมทุกความหมาย

3. Parse the sentence and output all trees using NLTK’s ChartParser.
วิเคราะห์ประโยคและปรินต์ syntax trees ทั้งหมดออกมาด้วย ChartParser ของ NLTK

4. Define a PCFG with assumed rule probabilities.
กำหนดน้ำหนักความน่าจะเป็นลงในกฎ เพื่อสร้างเป็น PCFG (สมมุติตัวเลขความน่าจะเป็นเองได้เลย ไม่ต้องใช้ Treebank )

5. Print the most probable parse tree and its probability using ViterbiParser.
หา syntax trees ที่มีโอกาสเกิดมากที่สุดพร้อมค่าความน่าจะเป็นด้วย ViterbiParser ของ NLTK

6. Display the dependency parse tree of the sentence.
แสดงผลแผนภาพ Dependency Tree ของประโยคนั้น






In [41]:
# TODO: import lib here
import random

import spacy
from spacy import displacy

import nltk
from nltk import CFG, PCFG, Nonterminal, Tree, induce_pcfg
from nltk.grammar import Production
from nltk.parse import ChartParser, ViterbiParser


random.seed(13)
nlp = spacy.load("en_core_web_sm")

print("nltk  :", nltk.__version__)
print("spaCy :", spacy.__version__)

nltk  : 3.10.3
spaCy : 3.8.16


In [42]:
## TODO: 1
SENT = "Students hate tests with errors"

In [43]:
grammar = CFG.fromstring("""
    S -> NP VP
    NP -> N | NP PP
    VP -> V NP | VP PP
    PP -> P NP
    V -> 'hate'
    P -> 'with'
    N -> 'students' | 'tests' | 'errors'
""")

SENT = "students hate tests with errors".split()

parser = ChartParser(grammar)

trees = list(parser.parse(SENT))
for tree in trees:
    tree.pretty_print()

                    S                      
    ________________|____                   
   |                     VP                
   |            _________|________          
   |           VP                 PP       
   |       ____|____          ____|____     
   NP     |         NP       |         NP  
   |      |         |        |         |    
   N      V         N        P         N   
   |      |         |        |         |    
students hate     tests     with     errors

                S                      
    ____________|____                   
   |                 VP                
   |       __________|___               
   |      |              NP            
   |      |      ________|____          
   |      |     |             PP       
   |      |     |         ____|____     
   NP     |     NP       |         NP  
   |      |     |        |         |    
   N      V     N        P         N   
   |      |     |        |         |    
students hate tests     

In [44]:
## TODO: 3

print("number of possible trees:", len(trees))
print()

## HINT: for loop print all possible trees
for i, tree in enumerate(trees):
    print(f"Tree {i + 1}:")
    tree.pretty_print()
    print()

number of possible trees: 2

Tree 1:
                    S                      
    ________________|____                   
   |                     VP                
   |            _________|________          
   |           VP                 PP       
   |       ____|____          ____|____     
   NP     |         NP       |         NP  
   |      |         |        |         |    
   N      V         N        P         N   
   |      |         |        |         |    
students hate     tests     with     errors


Tree 2:
                S                      
    ____________|____                   
   |                 VP                
   |       __________|___               
   |      |              NP            
   |      |      ________|____          
   |      |     |             PP       
   |      |     |         ____|____     
   NP     |     NP       |         NP  
   |      |     |        |         |    
   N      V     N        P         N   
   |      |     |  

In [45]:
PG = PCFG.fromstring("""
    S -> NP VP              [1.0]

    NP -> N_Students        [0.25]
    NP -> N_Tests           [0.25]
    NP -> NP PP             [0.25]
    NP -> N_Errors          [0.25]

    VP -> V NP              [0.5]
    VP -> VP PP             [0.5]

    PP -> P NP              [1.0]
    
    V -> 'hate'             [1.0]
    P -> 'with'             [1.0]
    N_Students -> 'students'[1.0]
    N_Tests -> 'tests'      [1.0]
    N_Errors -> 'errors'    [1.0]
""")

## check this first!
print("in CNF:", PG.is_chomsky_normal_form())

for lhs in [Nonterminal("NP"), Nonterminal("VP")]:
    ps = PG.productions(lhs=lhs)
    print(f"{lhs} probabilities sum to", round(sum(p.prob() for p in ps), 6))


in CNF: False
NP probabilities sum to 1.0
VP probabilities sum to 1.0


In [46]:
## TODO: 5
vp_parser = ViterbiParser(PG)

trees = list(vp_parser.parse(SENT))
best_path = trees[0]

best_prob = best_path.prob()

print("best path probability:", best_prob)


best path probability: 0.00390625


In [47]:
## TODO: 6
doc = nlp("students hate tests with errors")

print(f"{'i':>3} {'word':<12}{'POS':<7}{'dep':<10}{'head':<12}")
print("-" * 48)
for tok in doc:
    head = "ROOT" if tok.head.i == tok.i else f"{tok.head.text} ({tok.head.i})"
    print(f"{tok.i:>3} {tok.text:<12}{tok.pos_:<7}{tok.dep_:<10}{head:<12}")
    

  i word        POS    dep       head        
------------------------------------------------
  0 students    NOUN   nsubj     hate (1)    
  1 hate        VERB   ROOT      ROOT        
  2 tests       NOUN   dobj      hate (1)    
  3 with        ADP    prep      tests (2)   
  4 errors      NOUN   pobj      with (3)    
